In [1]:
import jax
import jax.numpy as jnp
from orbax.export import ExportManager, JaxModule, ServingConfig
from flax import nnx
from functools import partial
import numpy as np
import tensorflow_datasets as tfds  # TFDS to download MNIST.
import matplotlib.pyplot as plt

In [2]:
# Recreate the network class
# Note: this has to match exactly what was saved (to do: one can probably serialize the class object and avoid having to remember the exact structure)
class CNN(nnx.Module):
  """A simple CNN model."""

  def __init__(self, *, rngs: nnx.Rngs):
      
    self.conv1 = nnx.Conv(3, 32, kernel_size=(5, 5), rngs=rngs)
    self.dropout1 = nnx.Dropout(rate=0.3)
    self.batch_norm1 = nnx.BatchNorm(32, rngs=rngs)
    self.avg_pool1 = partial(nnx.avg_pool, window_shape=(4, 4), strides=(4, 4))

    self.conv2 = nnx.Conv(32, 64, kernel_size=(5, 5), rngs=rngs)
    self.batch_norm2 = nnx.BatchNorm(64, rngs=rngs)
    self.avg_pool2 = partial(nnx.avg_pool, window_shape=(8, 8), strides=(8, 8))

    self.linear1 = nnx.Linear(3136, 128, rngs=rngs) # 224 * 224 * 64 / [(4*4) * (8*8)]
    self.dropout2 = nnx.Dropout(rate=0.3)
    self.linear2 = nnx.Linear(128, 4, rngs=rngs)

  def __call__(self, x, rngs: nnx.Rngs | None = None):
    x = self.avg_pool1(nnx.relu(self.batch_norm1(self.dropout1(self.conv1(x), rngs=rngs))))
    x = self.avg_pool2(nnx.relu(self.batch_norm2(self.conv2(x))))
    x = x.reshape(x.shape[0], -1)  # flatten
    x = nnx.relu(self.dropout2(self.linear1(x), rngs=rngs))
    x = self.linear2(x)
    return x

# Instantiate the model.
cnn_model = CNN(rngs=nnx.Rngs(0))
# Visualize it.
nnx.display(cnn_model)

# View
eval_model = nnx.view(cnn_model, deterministic=True, use_running_average=True)


In [3]:
# Define the same prediction function used in export
def exported_predict(model, y):
    return model(y, None)

# Wrap your model again
jax_module = JaxModule(eval_model, exported_predict)

# Define input signature (same as when exported)
import tensorflow as tf
sig = [tf.TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32)]

output_dir = './cnn_export'
export_mgr = ExportManager(jax_module, [ServingConfig('mnist_server', input_signature=sig)])
loaded_cnn = export_mgr.load(output_dir)

In [4]:
# Example: run inference

# dummy input image
x = np.random.rand(1, 224, 224, 3).astype(np.float32)
loaded_cnn(x)


I0000 00:00:1774991635.748599    4194 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


<tf.Tensor: shape=(1, 4), dtype=float32, numpy=
array([[ 0.40687802, -0.03854495, -0.997021  ,  0.78146166]],
      dtype=float32)>

In [10]:
test_ds: tf.data.Dataset = tfds.load('galaxy_mnist', split='test', data_dir="./data")
test_ds = test_ds.map(
  lambda sample: {
    'image': tf.cast(sample['image'], tf.float32) / 255,
    'label': sample['label'],
  }
)
test_ds = test_ds.batch(1024, drop_remainder=True).prefetch(1)


In [11]:
# To do: figure out how to run batch inference on test_ds
'''
def predict_batch(x):
    # x shape: (B, 224, 224, 3)
    return jax.vmap(lambda img: loaded_cnn(img[None, ...]))(x) # this doesn't work, jax.vmap doesn't like tensorflow object

preds = predict_batch(test_batch['image'])
'''

"\ndef predict_batch(x):\n    # x shape: (B, 224, 224, 3)\n    return jax.vmap(lambda img: loaded_cnn(img[None, ...]))(x) # this doesn't work, jax.vmap doesn't like tensorflow object\n\npreds = predict_batch(test_batch['image'])\n"

In [12]:
'''
fig, axs = plt.subplots(5, 5, figsize=(12, 12))
for i, ax in enumerate(axs.flatten()):
  ax.imshow(test_batch['image'][i, ...], cmap='gray')
  ax.set_title(f'true/pred={test_batch["label"][i]}/{pred[i]}')
  ax.axis('off')
'''

'\nfig, axs = plt.subplots(5, 5, figsize=(12, 12))\nfor i, ax in enumerate(axs.flatten()):\n  ax.imshow(test_batch[\'image\'][i, ...], cmap=\'gray\')\n  ax.set_title(f\'true/pred={test_batch["label"][i]}/{pred[i]}\')\n  ax.axis(\'off\')\n'